In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [3]:
!pip install -q langchain-openai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 23.2 MB/s eta 0:00:00


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [5]:
#tools create
@tool
def multiply(a: int, b: int) -> int:
  """given two numbers a and b, this tool returns it's product"""
  return a*b

In [6]:
print(multiply.invoke({'a': 3, 'b': 4}))

12


Tool Binding

In [7]:
llm = ChatOpenAI()

In [8]:
llm_multiply = llm.bind_tools([multiply])

Tool Calling

In [9]:
query = 'hi how are you? can you please multiply 4 and 5'

In [10]:
messages = [HumanMessage(content=query)]

In [11]:
messages

[HumanMessage(content='hi how are you? can you please multiply 4 and 5', additional_kwargs={}, response_metadata={})]

In [12]:
result = llm_multiply.invoke(messages)

In [13]:
messages.append(result)

In [14]:
result.tool_calls[0]['args']

{'a': 4, 'b': 5}

Tool Execution

In [15]:
tool_result = multiply.invoke(result.tool_calls[0])

In [16]:
multiply.invoke({'name': 'multiply', 'args': {'a': 4, 'b': 5}, 'id': 'call_hgZdzcvwACnOdMy5J9houC66', 'type': 'tool_call'})

ToolMessage(content='20', name='multiply', tool_call_id='call_hgZdzcvwACnOdMy5J9houC66')

This above tool message can be sent to llm so that llm can generate the reply as well

In [17]:
messages.append(tool_result)

In [18]:
messages

[HumanMessage(content='hi how are you? can you please multiply 4 and 5', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 69, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ESEzVwrzc793vsghGwWGbRm3yEPop', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0dc20-8ca8-7490-83b9-0287a1552b4f-0', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 5}, 'id': 'call_OenYdo68o99wS99lpIWHETQz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 32, 'total_tokens':

In [19]:
llm_multiply.invoke(messages)

AIMessage(content='The product of 4 and 5 is 20.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 94, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ESEzXoJ1Eu8LKqpn9BBmOGscsijVJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0dc20-9765-7863-87c8-4f152679e1ed-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 94, 'output_tokens': 13, 'total_tokens': 107, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [20]:
llm_multiply.invoke(messages).content

'The product of 4 and 5 is 20.'

# Currency Conversion Tool

In [21]:
#tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated
@tool
def get_conversion_factor(base_currency: str , target_currency : str )-> dict:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c12ec68d8ddc9f628a2e60d3/pair/{base_currency}/{target_currency}'
  response = requests.get(url)
  return response.json()


@tool
def convert(base_currency_value:int, conversion_rate: Annotated[float,InjectedToolArg])-> float:
  """
given a currency conversion rate this function calculates the target currency value from a given base currency value
  """
  return base_currency_value * conversion_rate

In [22]:
get_conversion_factor.invoke({'base_currency': 'USD' , 'target_currency': 'INR' })

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1790380802,
 'time_last_update_utc': 'Sat, 26 Sep 2026 00:00:02 +0000',
 'time_next_update_unix': 1790467202,
 'time_next_update_utc': 'Sun, 27 Sep 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.9187}

In [23]:
convert.invoke({'base_currency_value': 10, 'conversion_rate': 85.16})

851.5999999999999

tool binding

In [24]:
llm1 = ChatOpenAI()

In [25]:
llm_with_tools = llm1.bind_tools([get_conversion_factor, convert])

Tool calling

In [26]:
messages = [HumanMessage('what is the conversion rate between usd and inr and based on that can you convert 19 usd to inr')]

In [27]:
messages

[HumanMessage(content='what is the conversion rate between usd and inr and based on that can you convert 19 usd to inr', additional_kwargs={}, response_metadata={})]

In [28]:
ai_message = llm_with_tools.invoke(messages)

In [29]:
messages.append(ai_message)

In [30]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_9de0NC8LVDLoa7hmdpRJ5Gci',
  'type': 'tool_call'}]

In [31]:
import json
for tool_call in ai_message.tool_calls:

  #execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':

    tool_message1 = get_conversion_factor.invoke(tool_call)

    #fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']

    print(conversion_rate)

    #append this tool message to messages list
    messages.append(tool_message1)

 #execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':

    #fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate

    #execute the tool
    tool_message2 = convert.invoke(tool_call)

    messages.append(tool_message2)


95.9187


In [32]:
messages

[HumanMessage(content='what is the conversion rate between usd and inr and based on that can you convert 19 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 123, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ESEzaiqlJvpJofI8eOgAThRcSwEqz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0dc20-a2e2-7110-9bbc-65f4fc6bc83d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_9de0NC8LVDLoa7hmdpRJ5Gci', 'type': 'tool_call'}],

In [34]:
while True:

  ai_message = llm_with_tools.invoke(messages)

  messages.append(ai_message)

  if not ai_message.tool_calls:
    print(ai_message.content)
    break

  for tool_call in ai_message.tool_calls:

    if tool_call['name'] == 'get_conversion_factor':

      tool_message1 = get_conversion_factor.invoke(tool_call)

      conversion_rate = json.loads(
          tool_message1.content
      )['conversion_rate']

      print(conversion_rate)

      messages.append(tool_message1)


    if tool_call['name'] == 'convert':

      tool_call['args']['conversion_rate'] = conversion_rate

      tool_message2 = convert.invoke(tool_call)

      messages.append(tool_message2)

The conversion rate between USD and INR is 1 USD to 95.9187 INR. Therefore, 19 USD is equivalent to 1822.4553 INR.
